Comparing Gemini annotations with BiomedParse pseudo-labels
1. Load the existing BiomedParse pseudo-label masks.
2. Identify all connected components without size filtering.
3. Calculate the area, equivalent diameter, maximum and minimum diameters, and bounding-box width and height.
4. Save the measurements to a CSV file.
5. Add Gemini measurements and compare the results across models.

In [ ]:
from google.colab import drive
from pathlib import Path
import pandas as pd
import numpy as np
import json

drive.mount("/content/drive")

# Directory containing the foundation model experiment results
PROJECT_DIR = Path(
    "/content/drive/MyDrive/ovarian_ultrasound_vlm"
)

MANIFEST_PATH = PROJECT_DIR / "image_manifest.csv"
PREDICTIONS_DIR = PROJECT_DIR / "predictions"

# Load the dataset manifest
manifest = pd.read_csv(MANIFEST_PATH)

# Select the 10 pilot images
pilot = manifest[
    manifest["is_pilot"].astype(str).str.lower() == "true"
].copy()

pilot = pilot.sort_values("case_id").reset_index(drop=True)

# Verify that all pilot annotations are available
assert len(pilot) == 10

for case_id in pilot["case_id"]:
    prediction_path = PREDICTIONS_DIR / f"{case_id}.json"
    assert prediction_path.exists(), f"Missing prediction: {case_id}"

print("Total dataset images:", len(manifest))
print("Pilot images:", len(pilot))
print("Available Gemini annotations:", len(pilot))

display(
    pilot[
        ["case_id", "category", "split", "is_pilot"]
    ]
)

Mount Google Drive and load the dataset manifest for pilot evaluation

In [ ]:
from PIL import Image
import numpy as np

for _, row in pilot.iterrows():

    case_id = row["case_id"]

    image = np.array(Image.open(row["image_path"]))
    mask = np.array(Image.open(row["pseudo_label_path"]))

    # Check spatial dimensions
    assert image.shape[:2] == mask.shape[:2], (
        f"Dimension mismatch: {case_id}"
    )

    unique_values = np.unique(mask)

    print(
        f"{case_id}: "
        f"image={image.shape}, "
        f"mask={mask.shape}, "
        f"labels={unique_values}"
    )

Inspect the BiomedParse pseudo-label masks and verify their dimensions